# 01. Model Preparation (모델 학습 준비)

## 목적
1. Train/Val/Test 분할 (환자 단위 - Data Leakage 방지)
2. 피처/레이블 분리
3. 클래스 불균형 확인 및 가중치 계산

## 입력
- `features_final.csv`: FE 완료된 최종 피처셋

## 출력
- `train.csv`, `val.csv`, `test.csv`: 분할된 데이터셋
- `feature_cols.json`: 피처 목록
- `scale_pos_weights.json`: 클래스 불균형 가중치
- `split_info.json`: 분할 정보 (재현성 확인용)

## 최종 피처 구성 (35개)
| 구분 | 피처 | 개수 |
|------|------|------|
| Vitals | hr, rr, spo2, sbp, dbp, mbp, temp | 7 |
| Vitals 통계 | hr_max, rr_max, spo2_min, sbp_min | 4 |
| Labs | creatinine, wbc, platelets, potassium, sodium, lactate | 6 |
| GCS | gcs_eye, gcs_verbal, gcs_motor, gcs_total | 4 |
| Urine | urine_ml_6h, urine_ml_kg_hr_avg, oliguria_flag | 3 |
| 파생 | shock_index, anchor_age, news_score, mews_score | 4 |
| Flags | lactate_missing, abga_checked, is_readmission | 3 |
| Delta | hr_delta, sbp_delta, spo2_delta, lactate_delta, gcs_total_delta | 5 |

## 데이터 분할
- **Train**: 70%
- **Validation**: 15%
- **Test**: 15%
- **random_state=42** (팀 전체 일관성 유지)

In [ ]:
import pandas as pd
import numpy as np
import os
import json
from sklearn.model_selection import train_test_split

# ==============================================================================
# 설정
# ==============================================================================

INPUT_DIR = '../../data-pipeline/data/processed'
RANDOM_STATE = 42  # ⚠️ 절대 변경 X

# ------------------------------------------------------------------------------
# 피처 선택 옵션
# ------------------------------------------------------------------------------
# 'all'     : 전체 35개 피처 사용
# 'top20'   : SHAP 상위 20개 피처 사용 (별도 json 필요)
# 'top10'   : SHAP 상위 10개 피처 사용 (별도 json 필요)
# 'custom'  : 직접 지정
# ------------------------------------------------------------------------------
FEATURE_CONFIG = 'all'  # 여기만 변경하세요

print("=== 01. Model Preparation 시작 ===")
print(f"\n설정:")
print(f"  - RANDOM_STATE: {RANDOM_STATE}")
print(f"  - FEATURE_CONFIG: {FEATURE_CONFIG}")

## Step 1: 데이터 로드

In [ ]:
print("\nStep 1: 데이터 로드")

df = pd.read_csv(os.path.join(INPUT_DIR, 'features_final.csv'))

print(f"✓ 데이터 로드 완료: {len(df):,} rows")
print(f"  - 고유 환자 수: {df['stay_id'].nunique():,}명")
print(f"  - 컬럼 수: {len(df.columns)}개")

## Step 2: 피처/레이블 정의

### 35개 피처 정의
SHAP 분석 및 임상적 근거 기반으로 선정된 피처

In [ ]:
print("\nStep 2: 피처/레이블 정의")

# ==============================================================================
# ID 컬럼 (모델 학습 제외)
# ==============================================================================
id_cols = [
    'stay_id',           # ICU 체류 ID
    'subject_id',        # 환자 ID
    'hadm_id',           # 입원 ID
    'observation_hour',  # 관찰 시점 (ICU 입실 후 시간)
    'observation_start', # 관찰 윈도우 시작
    'observation_end'    # 관찰 윈도우 종료
]

# ==============================================================================
# 레이블 컬럼 (예측 대상)
# ==============================================================================
label_cols = [col for col in df.columns if 'next_' in col]

# ==============================================================================
# 피처 컬럼 (35개)
# ==============================================================================
# 전체 피처 정의 (순서 유지)
ALL_FEATURES = {
    'vitals': ['hr', 'rr', 'spo2', 'sbp', 'dbp', 'mbp', 'temp'],
    'vitals_stat': ['hr_max', 'rr_max', 'spo2_min', 'sbp_min'],
    'labs': ['creatinine', 'wbc', 'platelets', 'potassium', 'sodium', 'lactate'],
    'gcs': ['gcs_eye', 'gcs_verbal', 'gcs_motor', 'gcs_total'],
    'urine': ['urine_ml_6h', 'urine_ml_kg_hr_avg', 'oliguria_flag'],
    'derived': ['shock_index', 'anchor_age', 'news_score', 'mews_score'],
    'flags': ['lactate_missing', 'abga_checked', 'is_readmission'],
    'delta': ['hr_delta', 'sbp_delta', 'spo2_delta', 'lactate_delta', 'gcs_total_delta']
}

# 전체 피처 리스트 생성
all_feature_list = []
for group, cols in ALL_FEATURES.items():
    all_feature_list.extend(cols)

print(f"\n전체 피처 수: {len(all_feature_list)}개")
# print(f"\n전체 피처 리스트: {all_feature_list}")

In [ ]:
# ==============================================================================
# 피처 선택
# ==============================================================================

if FEATURE_CONFIG == 'all':
    # 전체 35개 피처 사용
    feature_cols = all_feature_list
    OUTPUT_DIR = os.path.join(INPUT_DIR, 'all_features')
    
elif FEATURE_CONFIG.startswith('top'):
    # SHAP 기반 상위 N개 피처 (별도 json 필요)
    json_path = os.path.join(INPUT_DIR, f'{FEATURE_CONFIG}_features.json')
    if os.path.exists(json_path):
        with open(json_path, 'r') as f:
            feature_cols = json.load(f)
    else:
        raise FileNotFoundError(f"{json_path} 파일이 없습니다. SHAP 분석 후 생성하세요.")
    OUTPUT_DIR = os.path.join(INPUT_DIR, FEATURE_CONFIG)
    
elif FEATURE_CONFIG == 'custom':
    # 직접 지정
    feature_cols = [
        # 여기에 원하는 피처 직접 입력
        'hr', 'sbp', 'spo2', 'lactate', 'gcs_total',
        'news_score', 'mews_score', 'shock_index', 'anchor_age'
    ]
    OUTPUT_DIR = os.path.join(INPUT_DIR, 'custom_features')
    
else:
    raise ValueError(f"Unknown FEATURE_CONFIG: {FEATURE_CONFIG}")

# 출력 디렉토리 생성
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 존재하는 피처만 필터링
feature_cols = [col for col in feature_cols if col in df.columns]

print(f"\n선택된 피처: {len(feature_cols)}개")
print(f"출력 디렉토리: {OUTPUT_DIR}")

In [ ]:
# 피처 존재 여부 확인
print("\n=== 피처 존재 확인 ===")
missing_features = [col for col in feature_cols if col not in df.columns]

if missing_features:
    print(f"⚠️ 없는 피처: {missing_features}")
else:
    print(f"✓ 모든 피처 존재")

# 그룹별 피처 현황
print("\n=== 피처 그룹별 현황 ===")
for group, cols in ALL_FEATURES.items():
    selected = [c for c in cols if c in feature_cols]
    print(f"  [{group}]: {len(selected)}/{len(cols)}개")
    if selected:
        print(f"    → {selected}")

In [ ]:
# 레이블 분포 확인
print("\n=== 레이블 목록 ===")
for col in sorted(label_cols):
    pos_rate = df[col].mean() * 100
    pos_count = df[col].sum()
    print(f"  {col}: {pos_count:,} ({pos_rate:.2f}%)")

## Step 3: Train/Val/Test 분할 (환자 단위)

### ⚠️ 중요: Data Leakage 방지
- **행 단위가 아닌 "환자 단위"로 분할**
- 같은 환자가 train과 test에 동시에 있으면 cheating
- 환자 A의 6h 데이터로 학습 → 환자 A의 12h 예측 → 당연히 잘 맞음 (누수!)
- 실제 배포 시에는 "처음 보는 환자"를 예측해야 함

### 분할 비율
- Train: 70%
- Validation: 15%
- Test: 15%

### 재현성
- **random_state=42 고정**
- 팀원 모두 동일한 분할 결과

In [ ]:
print("\nStep 3: Train/Val/Test 분할 (환자 단위)")

# --- 고유 환자 추출 ---
unique_patients = df['subject_id'].unique()  # stay_id → subject_id
print(f"  총 환자 수: {len(unique_patients):,}명")
print(f"  총 stay 수: {df['stay_id'].nunique():,}개")

# --- 1차 분할: Train (70%) vs Temp (30%) ---
train_patients, temp_patients = train_test_split(
    unique_patients,
    test_size=0.30,
    random_state=RANDOM_STATE
)

# --- 2차 분할: Val (15%) vs Test (15%) ---
val_patients, test_patients = train_test_split(
    temp_patients,
    test_size=0.50,
    random_state=RANDOM_STATE
)

# --- 환자 기준으로 데이터 분할 ---
train = df[df['subject_id'].isin(train_patients)]  # stay_id → subject_id
val = df[df['subject_id'].isin(val_patients)]
test = df[df['subject_id'].isin(test_patients)]

print(f"\n  환자 수 분할:")
print(f"    - Train: {len(train_patients):,}명 ({len(train_patients)/len(unique_patients)*100:.1f}%)")
print(f"    - Val: {len(val_patients):,}명 ({len(val_patients)/len(unique_patients)*100:.1f}%)")
print(f"    - Test: {len(test_patients):,}명 ({len(test_patients)/len(unique_patients)*100:.1f}%)")

print(f"\n  샘플 수 분할:")
print(f"    - Train: {len(train):,}개 ({len(train)/len(df)*100:.1f}%)")
print(f"    - Val: {len(val):,}개 ({len(val)/len(df)*100:.1f}%)")
print(f"    - Test: {len(test):,}개 ({len(test)/len(df)*100:.1f}%)")

In [ ]:
# --- DataFrame 분할 ---
df_train = df[df['subject_id'].isin(train_patients)].copy()
df_val = df[df['subject_id'].isin(val_patients)].copy()
df_test = df[df['subject_id'].isin(test_patients)].copy()

print(f"  행 수 분할:")
print(f"    - Train: {len(df_train):,} rows ({len(df_train)/len(df)*100:.1f}%)")
print(f"    - Val: {len(df_val):,} rows ({len(df_val)/len(df)*100:.1f}%)")
print(f"    - Test: {len(df_test):,} rows ({len(df_test)/len(df)*100:.1f}%)")

# --- 분할 검증 ---
train_set = set(train_patients)
val_set = set(val_patients)
test_set = set(test_patients)

assert len(train_set & val_set) == 0, "Train-Val 환자 중복!"
assert len(train_set & test_set) == 0, "Train-Test 환자 중복!"
assert len(val_set & test_set) == 0, "Val-Test 환자 중복!"
print(f"\n✓ 환자 중복 없음 확인 완료")

## Step 4: 클래스 불균형 확인

### 예상 불균형
- death_next_6h: ~0.2% (매우 심각)
- composite_next_24h: ~4% (중간)

### 대응 방안
- XGBoost/LightGBM: `scale_pos_weight` 파라미터
- 평가 지표: AUROC, AUPRC (Accuracy 사용 금지)

In [ ]:
print("\nStep 4: 클래스 불균형 확인")

print("\n=== Train 세트 레이블 분포 ===")
imbalance_data = []

for col in sorted(label_cols):
    pos_count = df_train[col].sum()
    neg_count = len(df_train) - pos_count
    pos_rate = df_train[col].mean() * 100
    imbalance_ratio = neg_count / pos_count if pos_count > 0 else float('inf')
    
    # 심각도 표시
    if pos_rate < 1:
        severity = "🔴 심각"
    elif pos_rate < 3:
        severity = "🟡 주의"
    else:
        severity = "🟢 양호"
    
    imbalance_data.append({
        'label': col,
        'positive': pos_count,
        'negative': neg_count,
        'pos_rate': pos_rate,
        'ratio': imbalance_ratio
    })
    
    print(f"  {col}:")
    print(f"    Positive: {pos_count:,} ({pos_rate:.2f}%)")
    print(f"    Imbalance: 1:{imbalance_ratio:.0f} {severity}")

## Step 5: scale_pos_weight 계산

### XGBoost/LightGBM 클래스 가중치
```python
scale_pos_weight = negative_count / positive_count
```
- Positive 클래스에 가중치 부여
- 모델이 소수 클래스를 더 중요하게 학습

In [ ]:
print("\nStep 5: scale_pos_weight 계산")

scale_pos_weights = {}

for col in label_cols:
    pos_count = df_train[col].sum()
    neg_count = len(df_train) - pos_count
    
    if pos_count > 0:
        scale_pos_weights[col] = round(neg_count / pos_count, 1)
    else:
        scale_pos_weights[col] = 1.0

print("\n=== scale_pos_weight (XGBoost/LightGBM용) ===")
for col in sorted(scale_pos_weights.keys()):
    print(f"  {col}: {scale_pos_weights[col]}")

## Step 6: 저장

### 저장 파일 목록
| 파일 | 설명 |
|------|------|
| `train.csv` | 학습 데이터 (70%) |
| `val.csv` | 검증 데이터 (15%) |
| `test.csv` | 테스트 데이터 (15%) |
| `feature_cols.json` | 사용된 피처 목록 |
| `scale_pos_weights.json` | 클래스 불균형 가중치 |
| `split_info.json` | 분할 정보 (재현성 검증용) |

In [ ]:
print("\n" + "="*60)
print("Step 6: 저장")
print("="*60)

# --- CSV 저장 ---
train_path = os.path.join(OUTPUT_DIR, 'train.csv')
val_path = os.path.join(OUTPUT_DIR, 'val.csv')
test_path = os.path.join(OUTPUT_DIR, 'test.csv')

df_train.to_csv(train_path, index=False)
df_val.to_csv(val_path, index=False)
df_test.to_csv(test_path, index=False)

print(f"\n✓ 데이터 저장:")
print(f"  - train.csv: {len(df_train):,} rows, {os.path.getsize(train_path)/1024/1024:.1f} MB")
print(f"  - val.csv: {len(df_val):,} rows, {os.path.getsize(val_path)/1024/1024:.1f} MB")
print(f"  - test.csv: {len(df_test):,} rows, {os.path.getsize(test_path)/1024/1024:.1f} MB")

In [ ]:
# --- 피처 목록 저장 ---
feature_path = os.path.join(OUTPUT_DIR, 'feature_cols.json')
with open(feature_path, 'w') as f:
    json.dump(feature_cols, f, indent=2)
print(f"\n✓ 피처 목록 저장: feature_cols.json ({len(feature_cols)}개)")

# --- scale_pos_weight 저장 ---
weights_path = os.path.join(OUTPUT_DIR, 'scale_pos_weights.json')
with open(weights_path, 'w') as f:
    json.dump(scale_pos_weights, f, indent=2)
print(f"✓ 클래스 가중치 저장: scale_pos_weights.json")

# --- 분할 정보 저장 (재현성 검증용) ---
split_info = {
    'random_state': RANDOM_STATE,
    'feature_config': FEATURE_CONFIG,
    'n_features': len(feature_cols),
    'n_labels': len(label_cols),
    'total_samples': len(df),
    'total_patients': len(unique_patients),
    'train': {
        'n_patients': len(train_patients),
        'n_samples': len(df_train),
        'first_patient_id': int(train_patients[0]),  # 검증용
        'last_patient_id': int(train_patients[-1])
    },
    'val': {
        'n_patients': len(val_patients),
        'n_samples': len(df_val),
        'first_patient_id': int(val_patients[0]),
        'last_patient_id': int(val_patients[-1])
    },
    'test': {
        'n_patients': len(test_patients),
        'n_samples': len(df_test),
        'first_patient_id': int(test_patients[0]),
        'last_patient_id': int(test_patients[-1])
    }
}

split_info_path = os.path.join(OUTPUT_DIR, 'split_info.json')
with open(split_info_path, 'w') as f:
    json.dump(split_info, f, indent=2)
print(f"✓ 분할 정보 저장: split_info.json")

In [ ]:
# 저장된 파일 목록
print(f"\n=== 저장된 파일 목록 ===")
print(f"경로: {OUTPUT_DIR}")
for f in os.listdir(OUTPUT_DIR):
    fpath = os.path.join(OUTPUT_DIR, f)
    fsize = os.path.getsize(fpath) / 1024  # KB
    if fsize > 1024:
        print(f"  - {f}: {fsize/1024:.1f} MB")
    else:
        print(f"  - {f}: {fsize:.1f} KB")

In [ ]:
print("\n" + "="*60)
print("=== 01. Model Preparation 완료 ===")
print("="*60)

print(f"\n📋 요약:")
print(f"  - 피처: {len(feature_cols)}개")
print(f"  - 레이블: {len(label_cols)}개")
print(f"  - Train: {len(df_train):,} samples ({len(train_patients):,} patients)")
print(f"  - Val: {len(df_val):,} samples ({len(val_patients):,} patients)")
print(f"  - Test: {len(df_test):,} samples ({len(test_patients):,} patients)")
print(f"\n⚠️ RANDOM_STATE={RANDOM_STATE} 고정 - 팀원 모두 동일한 분할")
print(f"\n다음 단계: 02_train_xgboost.ipynb 또는 02_train_lightgbm.ipynb")

---
## 검증: 분할 재현성 확인

팀원이 동일한 분할 결과를 얻었는지 확인하는 코드

In [ ]:
# 분할 재현성 검증
print("=== 분할 재현성 검증 ===")
print(f"\n이 값들이 팀원 모두 동일해야 합니다:")
print(f"  - Train 첫 환자 ID: {train_patients[0]}")
print(f"  - Train 마지막 환자 ID: {train_patients[-1]}")
print(f"  - Val 첫 환자 ID: {val_patients[0]}")
print(f"  - Test 첫 환자 ID: {test_patients[0]}")
print(f"\n위 값이 다르면 random_state 또는 입력 데이터가 다른 것입니다.")

## 참고: 팀원용 데이터 로드 코드

```python
import pandas as pd
import json

# 데이터 로드
DATA_DIR = '../data/processed/all_features'  # 또는 top20, custom 등

train = pd.read_csv(f'{DATA_DIR}/train.csv')
val = pd.read_csv(f'{DATA_DIR}/val.csv')
test = pd.read_csv(f'{DATA_DIR}/test.csv')

# 피처 목록 로드
with open(f'{DATA_DIR}/feature_cols.json', 'r') as f:
    feature_cols = json.load(f)

# 클래스 가중치 로드
with open(f'{DATA_DIR}/scale_pos_weights.json', 'r') as f:
    scale_pos_weights = json.load(f)

# X, y 분리
X_train = train[feature_cols]
y_train = train['composite_next_24h']  # 원하는 레이블

X_val = val[feature_cols]
y_val = val['composite_next_24h']

X_test = test[feature_cols]
y_test = test['composite_next_24h']
```